# LLM Gateways Explained - Built One with LiteLLM + LangChain

1. **What is an LLM Gateway ?** - The problem it solves
2. **Why do we need it ?** - Real production pain points
3. **Core capabilities** - Routing, fallbacks, caching, observability, cost tracking
4. **Practical implementation** - Build one from scratch using `LiteLLM`
5. **Integration with LangChain** - Plu the getway into your agentic apps
6. **Production patterns** - Logging, retries, multi-provider fallbacks

By the end, we'll have a **Working LLM gateway** that routes between OpenAI, Anthropic, and Groq with built in.

## Part 1: What is and LLM Gateway ?

Think of an **LLM Gateway** as a **smart middleware layer** that sits between your application and multiple LLM providers (OpnAI, Anthropic, Google, Groq, Cohere, local models, etc.).

```mermaid
flowchart LR
    subgraph Provider [" "]
        direction LR
        O["OpenAI"]
        C["Claude"]
        G["Gemini"]
        R["Groq"]
    end

A --> B
B --> Provider


A["<div style='height:70px; display:flex; flex-direction:column; justify-content:center; align-items:center;'><b>Your Application</b><hr style='width:90%; margin:4px 0;'/></br>(Chatbot, RAG, Agent, etc)</div>"]

B["<div style='display:flex; flex-direction:column; justify-content:center; align-items:center;'><b>LLM GATEWAY</b><hr style='width:90%; margin:4px 0;'/><ul style='text-align:left; margin:0; padding-left:20px;'><li>Routing</li><li>Fallbacks</li><li>Caching</li><li>Rate Limiting</li><li>Cost Tracking</li><li>Observability</li></ul></div>"]


```

## Without a Gateway (The Pain 😔)

- Different SDKs and APIs for every provider
- No fallback if one provider goes down
- No central place to track costs
- Hard to switch models without rewriting code
- No caching -> paying twice for the same query

## With a Gateway (The Joy 😎)

- **One unified API** for 100+ providers
- **Automatic fallbacks** if a provider fails
- **Centralized logging, cost tracking, rate limiting**
- **Swap models with a config change**, no code rewrite
- **Cache repeated queries** -> save money

## Part 2: Installation & Setup

We'll use:

- **LiteLLM** -> the model popular open-source LLM gateway (supports 100+ providers)
- **LangChain** -> for building agentic workflows on top of the gateway
- **Python-dotnev** -> for managing API keys

In [ ]:
# install the required packages
!uv add litellm

In [5]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

# Now import LiteLLM normally
from litellm import completion

In [6]:
import litellm
litellm.suppress_debug_info = True

In [13]:
# Load API keys from a .env file
# Create a .env file in the. same folder with:
# GOOGLE_API_KEY
# GROQ_API_KEY

import os
from dotenv import load_dotenv
load_dotenv()

# Quick check
print("Gemini key loaded: ", "✅" if os.getenv("GOOGLE_API_KEY") else "❌")
print("OpenAI key loaded: ", "✅" if os.getenv("OpenAI_API_KEY") else "❌")
print("Groq key loaded:   ", "✅" if os.getenv("GROQ_API_KEY") else "❌")

Gemini key loaded:  ✅
OpenAI key loaded:  ❌
Groq key loaded:    ✅


## Part 3: The Simplest LiteLLM Example - Unified API

The biggest pain point: **every provider has different SDK.**

LiteLLM gives you **one function** - `completion()` - that works with all of them. Look at how clean this is:

In [49]:
from litellm import completion
# Same code, different providers - just change the `model` string!

# Call google gemini
response_gemini = completion(
    model="gemini/gemini-3.5-flash",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}]
)

print("♊️ Gemini: ", response_gemini.choices[0].message.content)

♊️ Gemini:  **Retrieval-Augmented Generation (RAG)** is an AI framework that improves the accuracy and reliability of large language models by fetching relevant facts from an external knowledge base before generating a response.


In [50]:
# Call groq (super fast inference)
response_groq = completion(
    model="groq/openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}]
)

print("✴️ Groq:   ", response_groq.choices[0].message.content)

✴️ Groq:    Retrieval‑Augmented Generation (RAG) is a technique that combines a language model with an external knowledge source, automatically fetching relevant documents and feeding them into the model so it can generate more accurate, up‑to‑date, and fact‑grounded responses.


but a real LLM Gateway does much more. Let's build those features one by one

In [51]:
from litellm import completion

prompt = "Explain RAG in one sentence."

# Just s alist of model strings - that's only configuration
providers = [
    ("✴️ Groq", "groq/openai/gpt-oss-120b"),
    ("♊️ Gemini", "gemini/gemini-3.5-flash"),
    ("☣️ OpenAI", "gpt-4o-mini"),
]

# ONE loop. ONE function call. Multiple providers.
for label, model in providers:
    try:
        r = completion(model=model, messages=[{"role": "user", "content": prompt}])
        print(f"{label:<15}: {r.choices[0].message.content[:80]}")
    except Exception as e:
        print(f"{label:<15}: ❌ {type(e).__name__}")

✴️ Groq        : Retrieval‑Augmented Generation (RAG) is a technique that combines a language mod
♊️ Gemini      : **Retrieval-Augmented Generation (RAG)** is an AI framework that improves the ac
☣️ OpenAI      : ❌ InternalServerError


## Part 4: Automatic Fallbacks - When OpenAI Goes Down

**Real story:** OpenAI had a 4-hour outable in November 2023. Apps that hard-coded `gpt-4` went completely dark. With a gateway, if one provider fails, we **automatically fall back** to another. Production apps must have this:

In [52]:
from litellm import completion

# Define a fallback chain: try GPT first, then gemini, then Grqo
response = completion(
    model="gpt-4o-mini",
    messages = [{"role": "user", "content": "What is an LLM Gateway?"}],
    fallbacks = [
        "gemini/gemini-3.5-flash",
        "groq/openai/gpt-oss-120b",
    ]
)

print("Response:", response.choices[0].message.content[:200], "...")
print("\nWhich model actually answered?", response.model)

19:23:52 - LiteLLM:ERROR: fallback_utils.py:75 - Fallback attempt failed for model gpt-4o-mini: litellm.InternalServerError: InternalServerError: OpenAIException - Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.
Traceback (most recent call last):
  File "/Users/rahulshelke/Documents/Data-Science/Data-Science-Projects/Complete-Agentic-AI-Course/.venv/lib/python3.13/site-packages/litellm/llms/openai/openai.py", line 852, in acompletion
    openai_aclient: AsyncOpenAI = self._get_openai_client(
                                  ~~~~~~~~~~~~~~~~~~~~~~~^
        is_async=True,
        ^^^^^^^^^^^^^^
    ...<7 lines>...
        shared_session=shared_session,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/Users/rahulshelke/Documents/Data-Science/Data-Science-Projects/Complete-Agentic-AI-Course/.venv/lib/python3.13/site-packages/litellm/llms/openai/openai.py", line 369, in 

Response: An **LLM Gateway** (also known as an AI Gateway) is a centralized proxy server that sits between your applications and the various Large Language Model (LLM) providers (such as OpenAI, Anthropic, Cohe ...

Which model actually answered? gemini-3.5-flash


if `gpt-4o-mini` is rate-limited or down, LiteLLM transparently ratries with Claude, then Groq. Your app **never see the failure.**

This is the #1 reason teams adopt an LLM Gateway.

In [53]:
from litellm import completion

# Force the primery to fail by useing a fake model name
# then watch the fallnack chain reuse the call
response = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What is an LLM Gateway?"}],
    fallbacks=[
        "gemini/gemini-3.5-flash",
        "groq/openai/gpt-oss-120b",
    ]
)

print("✅ App still got a response, even though the primery failed!")
print(f"\n🤖 Model that actually answered: {response.model}")
print(f"\n📄 Response: {response.choices[0].message.content[:200]}...")

✅ App still got a response, even though the primery failed!

🤖 Model that actually answered: gemini-3.5-flash

📄 Response: An **LLM Gateway** (also known as an AI Gateway) is a centralized proxy server that sits between your applications and the various Large Language Model (LLM) providers (such as OpenAI, Anthropic, Cohe...
